# 📊 Predicción de Carga Operativa — Backoffice Call Center Financiero Honduras
**Modelo:** XGBoost Multiclase (Bajo / Medio / Alto)  
**Horizonte:** Próximos 7 días  
**Versión:** 2.0 — Estacionalidad completa de negocio

### Factores de negocio incorporados
| Factor | Descripción |
|--------|-------------|
| 📅 Semana Santa | Mar/Abr — semana de vacación nacional, volumen cae drásticamente |
| 🇭🇳 Semana Morazánica | Oct — actividad reducida toda la semana |
| 🎄 Aguinaldo (Dic) | Mayor flujo de dinero → más pagos Y más mora por gasto excesivo |
| 📉 Enero | Resaca financiera de Dic → menos flujo, mora alta |
| 💵 14vo salario (Jun) | 1ra quincena: menos mora porque pagan; 2da: regresa a normal |
| ✂️ Fechas de corte | Días 5,10,15,20,23,25,28 → pico de solicitudes por mora |
| 📆 Cierre de mes | Días 27-31 → pico máximo de cobros y acuerdos |
| 📆 Inicio de mes | Días 1-5 → pico de gestiones post-cierre |
| 📅 Día de semana | Lunes alto (acumulado fin de semana), Viernes bajo |
| 🔔 Pre/Post feriado | Rebote de volumen el día siguiente a un feriado |

## 0. Instalación de dependencias

In [ ]:
# Descomenta y ejecuta solo la primera vez
# !pip install xgboost scikit-learn pandas numpy matplotlib seaborn openpyxl

## 1. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, accuracy_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Fechas de corte de clientes (días del mes que generan pico de mora)
FECHAS_CORTE = {5, 10, 15, 20, 23, 25, 28}

print('✅ Librerías importadas correctamente')

## 2. Carga de datos

In [ ]:
# ─── CONFIGURA AQUÍ ───────────────────────────────────────────────────────────
RUTA_DATOS    = 'datos_solicitudes.csv'
RUTA_FERIADOS = 'feriados_honduras.csv'
# ──────────────────────────────────────────────────────────────────────────────

def cargar_archivo(ruta):
    if ruta.endswith(('.xlsx', '.xls')):
        return pd.read_excel(ruta)
    return pd.read_csv(ruta, encoding='utf-8-sig')

df_raw = cargar_archivo(RUTA_DATOS)
df_raw.columns = (
    df_raw.columns.str.strip().str.upper()
    .str.replace(' ', '_').str.replace('(','',regex=False).str.replace(')','',regex=False)
)

# Mapeo de columnas al nombre estándar
MAPA = {
    'CREATED_DATE'          : 'CREATED_DATE',
    'SOLICITUD_ID'          : 'SOLICITUD_ID',
    'GESTION_TYPE'          : 'GESTION_TYPE',
    'PRODUCT_TYPETC_OR_PR'  : 'PRODUCT_TYPE',
    'DELIQUENCY'            : 'DELIQUENCY',
    'CYCLE'                 : 'CYCLE',
    'COUNTRY'               : 'COUNTRY',
}
df_raw.rename(columns=MAPA, inplace=True)

# Feriados
df_feriados = cargar_archivo(RUTA_FERIADOS)
df_feriados.columns = df_feriados.columns.str.strip().str.lower()
df_feriados['fecha'] = pd.to_datetime(df_feriados['fecha'])
FERIADOS_SET = set(df_feriados['fecha'].dt.date)

# Semanas especiales (para features de bloque de vacaciones)
# Se detectan automáticamente a partir del archivo de feriados
df_feriados['mes'] = df_feriados['fecha'].dt.month
df_feriados['anio'] = df_feriados['fecha'].dt.year

# Feriados de Semana Santa (mar/abr) → expandir ±3 días para capturar la semana completa
ss_dates = df_feriados[df_feriados['mes'].isin([3, 4])]['fecha']
SEMANA_SANTA_SET = set()
for d in ss_dates:
    for delta in range(-3, 5):
        SEMANA_SANTA_SET.add((d + pd.Timedelta(days=delta)).date())

# Feriados Morazánicos (octubre primeros días)
morazan_dates = df_feriados[(df_feriados['mes'] == 10) & (df_feriados['fecha'].dt.day <= 5)]['fecha']
MORAZAN_SET = set()
for d in morazan_dates:
    for delta in range(7):
        MORAZAN_SET.add((d + pd.Timedelta(days=delta)).date())

print(f'📂 Registros cargados   : {len(df_raw):,}')
print(f'📅 Feriados cargados    : {len(FERIADOS_SET)}')
print(f'🌿 Días Semana Santa    : {len(SEMANA_SANTA_SET)}')
print(f'🇭🇳 Días Semana Morazánica: {len(MORAZAN_SET)}')
df_raw.head(3)

## 3. Limpieza y agregación diaria

In [ ]:
df_raw['CREATED_DATE'] = pd.to_datetime(df_raw['CREATED_DATE'], errors='coerce')
df_raw['fecha'] = df_raw['CREATED_DATE'].dt.normalize()
df_raw.dropna(subset=['fecha'], inplace=True)

# Filtrar Honduras si hay más países
if df_raw['COUNTRY'].nunique() > 1:
    df_raw = df_raw[df_raw['COUNTRY'].str.upper().str.contains('HND|HONDURAS', na=False)]

# Agregación diaria
daily = (
    df_raw.groupby('fecha')
    .agg(
        total_solicitudes = ('SOLICITUD_ID', 'count'),
        pct_TC            = ('PRODUCT_TYPE', lambda x: (x.str.upper()=='TC').mean()),
        avg_deliquency    = ('DELIQUENCY',   lambda x: pd.to_numeric(x, errors='coerce').mean()),
        avg_cycle         = ('CYCLE',        lambda x: pd.to_numeric(x, errors='coerce').mean()),
        n_gestiones_tipo  = ('GESTION_TYPE', 'nunique'),
        pct_cobro         = ('GESTION_TYPE', lambda x: (x=='COBRO').mean()),
        pct_acuerdo       = ('GESTION_TYPE', lambda x: (x=='ACUERDO_PAGO').mean()),
        pct_pago_parcial  = ('GESTION_TYPE', lambda x: (x=='PAGO_PARCIAL').mean()),
    )
    .reset_index().sort_values('fecha')
)

# Rellenar rango completo (incluyendo fines de semana con 0)
rango = pd.date_range(daily['fecha'].min(), daily['fecha'].max(), freq='D')
daily = daily.set_index('fecha').reindex(rango).rename_axis('fecha').reset_index()
daily['total_solicitudes'].fillna(0, inplace=True)

print(f'📊 Rango: {daily["fecha"].min().date()} → {daily["fecha"].max().date()}')
print(f'📅 Total días (incluyendo no hábiles): {len(daily):,}')
daily.head()

## 4. Feature Engineering — Estacionalidad de negocio

Aquí se construyen todas las variables que capturan los patrones de negocio hondureño.

In [ ]:
def build_features(df, feriados_set, semana_santa_set, morazan_set, fechas_corte):
    d = df.copy()
    f = d['fecha']

    # ── 1. Temporales básicas ─────────────────────────────────────────────────
    d['dia_semana']     = f.dt.dayofweek          # 0=Lun, 6=Dom
    d['dia_mes']        = f.dt.day
    d['mes']            = f.dt.month
    d['trimestre']      = f.dt.quarter
    d['anio']           = f.dt.year
    d['semana_anio']    = f.dt.isocalendar().week.astype(int)
    d['es_fin_semana']  = (d['dia_semana'] >= 5).astype(int)
    d['es_lunes']       = (d['dia_semana'] == 0).astype(int)
    d['es_martes']      = (d['dia_semana'] == 1).astype(int)
    d['es_miercoles']   = (d['dia_semana'] == 2).astype(int)
    d['es_viernes']     = (d['dia_semana'] == 4).astype(int)

    # ── 2. Feriados ───────────────────────────────────────────────────────────
    d['es_feriado']   = f.dt.date.isin(feriados_set).astype(int)
    d['pre_feriado']  = f.apply(lambda x: int((x + pd.Timedelta(days=1)).date() in feriados_set))
    d['post_feriado'] = f.apply(lambda x: int((x - pd.Timedelta(days=1)).date() in feriados_set))

    # ── 3. Semana Santa (vacación nacional mar/abr) ───────────────────────────
    d['es_semana_santa']  = f.dt.date.isin(semana_santa_set).astype(int)
    # Pre-semana santa: 5 días antes (alta anticipación de pagos)
    d['pre_semana_santa'] = f.apply(
        lambda x: int(any((x + pd.Timedelta(days=i)).date() in semana_santa_set for i in range(1,6)))
    )

    # ── 4. Semana Morazánica (vacación oct) ───────────────────────────────────
    d['es_semana_morazan']  = f.dt.date.isin(morazan_set).astype(int)
    d['pre_semana_morazan'] = f.apply(
        lambda x: int(any((x + pd.Timedelta(days=i)).date() in morazan_set for i in range(1,4)))
    )

    # ── 5. Estacionalidad financiera ──────────────────────────────────────────
    # Diciembre: aguinaldo (más dinero circulando)
    d['es_dic']           = (d['mes'] == 12).astype(int)
    d['es_dic_aguinaldo'] = ((d['mes'] == 12) & (d['dia_mes'].between(10, 20))).astype(int)

    # Enero: resaca financiera
    d['es_enero']         = (d['mes'] == 1).astype(int)

    # Junio: 14vo salario (1ra quincena bajan moras, pagan)
    d['es_jun_14vo']      = ((d['mes'] == 6) & (d['dia_mes'] <= 15)).astype(int)
    d['es_jun_2da_qna']   = ((d['mes'] == 6) & (d['dia_mes'] > 15)).astype(int)

    # ── 6. Ciclo del mes (fechas de corte y cierres) ──────────────────────────
    # Distancia al corte más cercano (en días)
    d['dist_corte_min'] = d['dia_mes'].apply(
        lambda dm: min(abs(dm - c) for c in fechas_corte)
    )
    d['es_cerca_corte'] = (d['dist_corte_min'] <= 2).astype(int)
    d['es_dia_corte']   = d['dia_mes'].isin(fechas_corte).astype(int)

    # Inicio de mes (días 1-5): pico de cobros post-cierre
    d['es_inicio_mes']  = (d['dia_mes'] <= 5).astype(int)

    # Cierre de mes (días 27-31): pico máximo de solicitudes
    d['es_cierre_mes']  = (d['dia_mes'] >= 27).astype(int)

    # Quincena (días 10-17): pico secundario
    d['es_quincena']    = (d['dia_mes'].between(10, 17)).astype(int)

    # Posición en el mes (0-1, continua)
    d['pos_mes_norm']   = d['dia_mes'] / 31.0

    # Señal cíclica del mes (captura el patrón periódico de cortes)
    d['sin_dia_mes']    = np.sin(2 * np.pi * d['dia_mes'] / 31)
    d['cos_dia_mes']    = np.cos(2 * np.pi * d['dia_mes'] / 31)

    # Señal cíclica del mes del año (estacionalidad anual)
    d['sin_mes']        = np.sin(2 * np.pi * d['mes'] / 12)
    d['cos_mes']        = np.cos(2 * np.pi * d['mes'] / 12)

    # Señal cíclica del día de semana
    d['sin_diasem']     = np.sin(2 * np.pi * d['dia_semana'] / 7)
    d['cos_diasem']     = np.cos(2 * np.pi * d['dia_semana'] / 7)

    # ── 7. Lags ───────────────────────────────────────────────────────────────
    for lag in [1, 2, 3, 5, 7, 14, 21, 28]:
        d[f'lag_{lag}d'] = d['total_solicitudes'].shift(lag)

    # Mismo día de la semana pasada y hace 2 semanas
    d['lag_semana_1'] = d['total_solicitudes'].shift(7)
    d['lag_semana_2'] = d['total_solicitudes'].shift(14)

    # ── 8. Rolling statistics ─────────────────────────────────────────────────
    s = d['total_solicitudes'].shift(1)
    for w in [7, 14, 30]:
        d[f'roll_mean_{w}d'] = s.rolling(w).mean()
        d[f'roll_std_{w}d']  = s.rolling(w).std()
        d[f'roll_max_{w}d']  = s.rolling(w).max()
        d[f'roll_min_{w}d']  = s.rolling(w).min()

    # ── 9. Variables contextuales del día ─────────────────────────────────────
    for col in ['pct_TC','avg_deliquency','avg_cycle','n_gestiones_tipo',
                'pct_cobro','pct_acuerdo','pct_pago_parcial']:
        if col in d.columns:
            d[col].fillna(d[col].median(), inplace=True)

    return d


daily = build_features(daily, FERIADOS_SET, SEMANA_SANTA_SET, MORAZAN_SET, FECHAS_CORTE)

FEATURE_COLS = [c for c in daily.columns if c not in ['fecha','total_solicitudes','carga']]
print(f'✅ Features construidas: {len(FEATURE_COLS)}')
print(FEATURE_COLS)

## 5. Análisis exploratorio de estacionalidad

Antes de modelar, verificamos visualmente que los patrones de negocio estén presentes en los datos.

In [ ]:
dias_habiles = daily[daily['total_solicitudes'] > 0].copy()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Análisis de estacionalidad — Factores de negocio Honduras', fontsize=14, y=1.01)

NOMBRES_MES = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
               7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}
NOMBRES_DIA = {0:'Lun',1:'Mar',2:'Mié',3:'Jue',4:'Vie',5:'Sáb',6:'Dom'}

# 1. Por mes (estacionalidad anual)
ax = axes[0,0]
por_mes = dias_habiles.groupby('mes')['total_solicitudes'].mean()
colores_mes = ['#e74c3c' if m in [1,10] else '#2ecc71' if m in [6] else
               '#f39c12' if m in [12] else '#3498db' for m in por_mes.index]
bars = ax.bar([NOMBRES_MES[m] for m in por_mes.index], por_mes.values,
              color=colores_mes, edgecolor='white', linewidth=1)
ax.set_title('Promedio diario por mes')
ax.set_ylabel('Solicitudes')
ax.axhline(por_mes.mean(), linestyle='--', color='gray', alpha=0.6, label='Promedio global')
ax.legend(fontsize=8)
for bar, val in zip(bars, por_mes.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{val:.0f}', ha='center', va='bottom', fontsize=8)

# 2. Por día de semana
ax = axes[0,1]
por_dia = dias_habiles.groupby('dia_semana')['total_solicitudes'].mean()
por_dia = por_dia[por_dia.index <= 4]  # Solo días hábiles
ax.bar([NOMBRES_DIA[d] for d in por_dia.index], por_dia.values,
       color='#9b59b6', edgecolor='white', linewidth=1)
ax.set_title('Promedio por día de semana')
ax.set_ylabel('Solicitudes')
ax.axhline(por_dia.mean(), linestyle='--', color='gray', alpha=0.6)

# 3. Por día del mes (ciclo mensual)
ax = axes[0,2]
por_dia_mes = dias_habiles.groupby('dia_mes')['total_solicitudes'].mean()
colores_dm = ['#e74c3c' if d in FECHAS_CORTE else
              '#f39c12' if d >= 27 else '#3498db' for d in por_dia_mes.index]
ax.bar(por_dia_mes.index, por_dia_mes.values, color=colores_dm,
       edgecolor='white', linewidth=0.5, width=0.8)
ax.set_title('Promedio por día del mes')
ax.set_xlabel('Día del mes')
ax.set_ylabel('Solicitudes')
ax.axhline(por_dia_mes.mean(), linestyle='--', color='gray', alpha=0.6)
# Leyenda
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#e74c3c', label='Fecha de corte'),
    Patch(facecolor='#f39c12', label='Cierre de mes'),
    Patch(facecolor='#3498db', label='Día normal'),
], fontsize=8, loc='upper left')

# 4. Serie temporal completa
ax = axes[1,0]
ax.plot(dias_habiles['fecha'], dias_habiles['total_solicitudes'],
        color='#3498db', linewidth=0.8, alpha=0.7)
# Sombrear Semana Santa
for anio in [2022, 2023, 2024]:
    ss_anio = [d for d in SEMANA_SANTA_SET if d.year == anio]
    if ss_anio:
        ax.axvspan(pd.Timestamp(min(ss_anio)), pd.Timestamp(max(ss_anio)),
                   alpha=0.15, color='green', label='Semana Santa' if anio==2022 else '')
    # Octubre Morazánica
    mz = [d for d in MORAZAN_SET if d.year == anio]
    if mz:
        ax.axvspan(pd.Timestamp(min(mz)), pd.Timestamp(max(mz)),
                   alpha=0.15, color='orange', label='Semana Morazánica' if anio==2022 else '')
ax.set_title('Serie histórica con vacaciones marcadas')
ax.set_ylabel('Solicitudes')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\'%y'))
ax.legend(fontsize=8)

# 5. Aguinaldo vs Enero
ax = axes[1,1]
dic_data = dias_habiles[dias_habiles['mes']==12]['total_solicitudes']
ene_data = dias_habiles[dias_habiles['mes']==1]['total_solicitudes']
jun_data = dias_habiles[dias_habiles['mes']==6]['total_solicitudes']
ax.boxplot([ene_data, jun_data, dic_data],
           labels=['Enero\n(resaca)', 'Junio\n(14vo)', 'Diciembre\n(aguinaldo)'],
           patch_artist=True,
           boxprops=dict(facecolor='#ecf0f1'),
           medianprops=dict(color='#e74c3c', linewidth=2))
ax.set_title('Distribución: meses financieros clave')
ax.set_ylabel('Solicitudes diarias')

# 6. Efecto post-feriado
ax = axes[1,2]
post = dias_habiles[dias_habiles['post_feriado']==1]['total_solicitudes']
norm = dias_habiles[(dias_habiles['post_feriado']==0) & 
                    (dias_habiles['es_feriado']==0) &
                    (dias_habiles['pre_feriado']==0)]['total_solicitudes']
ax.boxplot([norm, post],
           labels=['Día normal', 'Post-feriado'],
           patch_artist=True,
           boxprops=dict(facecolor='#ecf0f1'),
           medianprops=dict(color='#3498db', linewidth=2))
ax.set_title('Efecto rebote post-feriado')
ax.set_ylabel('Solicitudes diarias')

plt.tight_layout()
plt.show()

## 6. Etiquetado de carga operativa

In [ ]:
# Umbrales basados en percentiles de días con actividad real
dias_activos = daily[daily['total_solicitudes'] > 0]['total_solicitudes']

UMBRAL_BAJO = dias_activos.quantile(0.33)
UMBRAL_ALTO = dias_activos.quantile(0.67)

print(f'Umbral BAJO  (P33): {UMBRAL_BAJO:.0f} solicitudes/día')
print(f'Umbral ALTO  (P67): {UMBRAL_ALTO:.0f} solicitudes/día')

def etiquetar_carga(n):
    if n == 0:          return 'BAJO'   # días no hábiles
    if n <= UMBRAL_BAJO: return 'BAJO'
    if n <= UMBRAL_ALTO: return 'MEDIO'
    return 'ALTO'

daily['carga'] = daily['total_solicitudes'].apply(etiquetar_carga)
print('\nDistribución de clases:')
print(daily[daily['total_solicitudes']>0]['carga'].value_counts())

## 7. División temporal Train / Validación

In [ ]:
FEATURE_COLS = [c for c in daily.columns if c not in ['fecha','total_solicitudes','carga']]

daily_clean = daily.dropna(subset=FEATURE_COLS).copy()

DIAS_VALIDACION = 90   # Últimos 3 meses como validación
fecha_corte = daily_clean['fecha'].max() - pd.Timedelta(days=DIAS_VALIDACION)

train = daily_clean[daily_clean['fecha'] <= fecha_corte].copy()
val   = daily_clean[daily_clean['fecha'] >  fecha_corte].copy()

le = LabelEncoder()
le.fit(['BAJO', 'MEDIO', 'ALTO'])

X_train = train[FEATURE_COLS]
y_train = le.transform(train['carga'])
X_val   = val[FEATURE_COLS]
y_val   = le.transform(val['carga'])

print(f'Corte    : {fecha_corte.date()}')
print(f'Train    : {len(train):,} días  →  {train["fecha"].min().date()} a {train["fecha"].max().date()}')
print(f'Val      : {len(val):,} días  →  {val["fecha"].min().date()} a {val["fecha"].max().date()}')
print(f'Features : {len(FEATURE_COLS)}')

## 8. Entrenamiento XGBoost

In [ ]:
modelo = XGBClassifier(
    objective             = 'multi:softprob',
    num_class             = 3,
    n_estimators          = 500,
    learning_rate         = 0.04,
    max_depth             = 5,
    min_child_weight      = 3,
    subsample             = 0.80,
    colsample_bytree      = 0.75,
    gamma                 = 0.1,
    reg_alpha             = 0.1,
    reg_lambda            = 1.0,
    use_label_encoder     = False,
    eval_metric           = 'mlogloss',
    early_stopping_rounds = 40,
    random_state          = RANDOM_STATE,
    verbosity             = 0,
)

modelo.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

print(f'\n✅ Mejor iteración: {modelo.best_iteration}')
print(f'   Mejor log-loss: {modelo.best_score:.4f}')

## 9. Evaluación del modelo

In [ ]:
y_pred        = modelo.predict(X_val)
y_pred_labels = le.inverse_transform(y_pred)
y_val_labels  = le.inverse_transform(y_val)

print(f'🎯 Accuracy en validación: {accuracy_score(y_val, y_pred):.2%}\n')
print(classification_report(y_val_labels, y_pred_labels,
                             target_names=['ALTO','BAJO','MEDIO']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Matriz de confusión
cm   = confusion_matrix(y_val_labels, y_pred_labels, labels=['BAJO','MEDIO','ALTO'])
disp = ConfusionMatrixDisplay(cm, display_labels=['BAJO','MEDIO','ALTO'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusión (validación)')

# Top 25 features por importancia
importancias = pd.Series(modelo.feature_importances_, index=FEATURE_COLS)
top25 = importancias.nlargest(25).sort_values()

colores_feat = []
for feat in top25.index:
    if any(k in feat for k in ['semana_santa','morazan']):
        colores_feat.append('#2ecc71')
    elif any(k in feat for k in ['cierre','inicio_mes','corte','quincena']):
        colores_feat.append('#e74c3c')
    elif any(k in feat for k in ['dic','enero','jun','aguinaldo']):
        colores_feat.append('#f39c12')
    elif any(k in feat for k in ['lag','roll']):
        colores_feat.append('#9b59b6')
    else:
        colores_feat.append('#3498db')

top25.plot(kind='barh', ax=axes[1], color=colores_feat)
axes[1].set_title('Top 25 Features por importancia')
axes[1].set_xlabel('F-score')

from matplotlib.patches import Patch
leyenda_feat = [
    Patch(facecolor='#2ecc71', label='Vacaciones nacionales'),
    Patch(facecolor='#e74c3c', label='Ciclo mensual (cortes/cierres)'),
    Patch(facecolor='#f39c12', label='Estacionalidad financiera'),
    Patch(facecolor='#9b59b6', label='Lags / Rolling stats'),
    Patch(facecolor='#3498db', label='Temporales / Feriados'),
]
axes[1].legend(handles=leyenda_feat, fontsize=8, loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# Comparación visual real vs predicho en validación
COLORES = {'BAJO':'#2ecc71', 'MEDIO':'#f39c12', 'ALTO':'#e74c3c'}

df_val_result = val[['fecha','total_solicitudes','carga']].copy()
df_val_result['pred'] = y_pred_labels
df_val_result['ok']   = df_val_result['carga'] == df_val_result['pred']

fig, ax = plt.subplots(figsize=(18, 4))
ax.plot(df_val_result['fecha'], df_val_result['total_solicitudes'],
        color='#bdc3c7', linewidth=1, zorder=1, label='Solicitudes reales')

for etiqueta, color in COLORES.items():
    mask = df_val_result['pred'] == etiqueta
    ax.scatter(df_val_result.loc[mask,'fecha'],
               df_val_result.loc[mask,'total_solicitudes'],
               label=f'Pred: {etiqueta}', color=color,
               s=55, marker='^', zorder=4, alpha=0.85)

# Marcar predicciones incorrectas
errores = df_val_result[~df_val_result['ok']]
ax.scatter(errores['fecha'], errores['total_solicitudes'],
           s=120, marker='x', color='black', zorder=5, linewidths=1.5, label='Error')

ax.set_title('Validación — Solicitudes reales (línea) vs nivel predicho (triángulos)', fontsize=12)
ax.set_ylabel('Solicitudes')
ax.set_xlabel('Fecha')
ax.legend(fontsize=8, ncol=5)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.tight_layout()
plt.show()

## 10. Predicción de los próximos 7 días

In [ ]:
def predecir_proximos_dias(modelo, daily_clean, feriados_set, semana_santa_set,
                            morazan_set, fechas_corte, le, feature_cols, n_dias=7):
    # Estimado de solicitudes por nivel (para alimentar lags de días futuros)
    media_por_nivel = {
        nivel: daily_clean[daily_clean['carga']==nivel]['total_solicitudes'].mean()
        for nivel in ['BAJO','MEDIO','ALTO']
    }

    COLS_CTX = ['fecha','total_solicitudes','pct_TC','avg_deliquency',
                'avg_cycle','n_gestiones_tipo','pct_cobro','pct_acuerdo','pct_pago_parcial']
    contexto = daily_clean[[c for c in COLS_CTX if c in daily_clean.columns]].copy()

    ultima_fecha = contexto['fecha'].max()
    resultados   = []

    for i in range(1, n_dias + 1):
        fecha_pred = ultima_fecha + pd.Timedelta(days=i)

        # Valores contextuales estimados con la ventana reciente
        nueva = {
            'fecha'             : fecha_pred,
            'total_solicitudes' : np.nan,
            'pct_TC'            : contexto['pct_TC'].iloc[-30:].mean(),
            'avg_deliquency'    : contexto['avg_deliquency'].iloc[-30:].mean(),
            'avg_cycle'         : contexto['avg_cycle'].iloc[-30:].mean(),
            'n_gestiones_tipo'  : contexto['n_gestiones_tipo'].iloc[-14:].mean(),
            'pct_cobro'         : contexto['pct_cobro'].iloc[-14:].mean(),
            'pct_acuerdo'       : contexto['pct_acuerdo'].iloc[-14:].mean(),
            'pct_pago_parcial'  : contexto['pct_pago_parcial'].iloc[-14:].mean(),
        }
        contexto = pd.concat([contexto, pd.DataFrame([nueva])], ignore_index=True)

        ctx_feat = build_features(contexto, feriados_set, semana_santa_set,
                                  morazan_set, fechas_corte)
        fila     = ctx_feat[ctx_feat['fecha'] == fecha_pred]

        if fila.empty:
            resultados.append({'fecha': fecha_pred, 'pred_carga': 'SIN DATOS'})
            continue

        X      = fila[feature_cols].fillna(method='ffill')
        proba  = modelo.predict_proba(X)[0]
        idx    = np.argmax(proba)
        pred   = le.inverse_transform([idx])[0]

        # Actualizar total_solicitudes con el estimado del nivel predicho
        contexto.loc[contexto['fecha'] == fecha_pred, 'total_solicitudes'] = media_por_nivel[pred]

        resultados.append({
            'fecha'       : fecha_pred,
            'dia'         : fecha_pred.strftime('%A'),
            'dia_mes'     : fecha_pred.day,
            'pred_carga'  : pred,
            'prob_bajo'   : round(proba[le.transform(['BAJO'])[0]], 3),
            'prob_medio'  : round(proba[le.transform(['MEDIO'])[0]], 3),
            'prob_alto'   : round(proba[le.transform(['ALTO'])[0]], 3),
            'es_feriado'  : int(fecha_pred.date() in feriados_set),
            'semana_santa': int(fecha_pred.date() in semana_santa_set),
            'morazanica'  : int(fecha_pred.date() in morazan_set),
        })

    return pd.DataFrame(resultados)


df_pred = predecir_proximos_dias(
    modelo, daily_clean, FERIADOS_SET, SEMANA_SANTA_SET,
    MORAZAN_SET, FECHAS_CORTE, le, FEATURE_COLS, n_dias=7
)

display(df_pred)

In [ ]:
# ── Visualización del pronóstico ──────────────────────────────────────────────
COLORES = {'BAJO':'#2ecc71', 'MEDIO':'#f39c12', 'ALTO':'#e74c3c', 'SIN DATOS':'#bdc3c7'}

fig, ax = plt.subplots(figsize=(13, 5))

etiquetas_x = [
    row['dia'][:3] + '\n' + str(int(row['dia_mes'])) +
    ('\n🔴 FERIADO' if row.get('es_feriado') else
     '\n🌿 SS' if row.get('semana_santa') else
     '\n🇭🇳 MZ' if row.get('morazanica') else '')
    if 'dia' in row else ''
    for _, row in df_pred.iterrows()
]

# Barras apiladas de probabilidades
x = range(len(df_pred))
ax.bar(x, df_pred['prob_alto'],  color='#e74c3c', label='P(ALTO)',  alpha=0.85)
ax.bar(x, df_pred['prob_medio'], bottom=df_pred['prob_alto'],
       color='#f39c12', label='P(MEDIO)', alpha=0.75)
ax.bar(x, df_pred['prob_bajo'],  bottom=df_pred['prob_alto']+df_pred['prob_medio'],
       color='#2ecc71', label='P(BAJO)',  alpha=0.65)

# Etiqueta del nivel predicho encima
for i, (_, row) in enumerate(df_pred.iterrows()):
    color_pred = COLORES.get(row.get('pred_carga','SIN DATOS'), '#555')
    ax.text(i, 1.04, row.get('pred_carga','?'),
            ha='center', va='bottom', fontsize=10, fontweight='bold', color=color_pred)

ax.set_xticks(list(x))
ax.set_xticklabels(etiquetas_x, fontsize=9)
ax.set_ylim(0, 1.18)
ax.set_ylabel('Probabilidad acumulada')
ax.set_title('📅 Pronóstico de carga operativa — Próximos 7 días', fontsize=13, pad=12)
ax.legend(loc='lower right', fontsize=9)
ax.axhline(0.5, linestyle='--', color='gray', alpha=0.4, linewidth=0.8)
plt.tight_layout()
plt.show()

## 11. Exportar resultados

In [ ]:
df_pred.to_csv('pronostico_7dias.csv', index=False)
daily_clean[['fecha','total_solicitudes','carga']].to_csv('historico_etiquetado.csv', index=False)
modelo.save_model('modelo_xgb_carga_v2.json')

print('✅ Exportados:')
print('   pronostico_7dias.csv')
print('   historico_etiquetado.csv')
print('   modelo_xgb_carga_v2.json')

---
## 📝 Próximos pasos recomendados

| Prioridad | Paso | Descripción |
|-----------|------|-------------|
| 🔴 Alta | **Ajustar umbrales** | Redefine BAJO/MEDIO/ALTO según tu capacidad real de agentes por turno |
| 🔴 Alta | **Validar fechas de corte** | Confirma que `{5,10,15,20,23,25,28}` son los cortes reales de tu cartera |
| 🟡 Media | **Agregar campañas** | Si hay campañas masivas de cobro, agrégalas como feature binaria |
| 🟡 Media | **Hyperparameter tuning** | Usa `Optuna` con `TimeSeriesSplit` para optimizar el modelo |
| 🟢 Baja | **Monitoreo de drift** | Compara distribución de features mensualmente con `evidently` |
| 🟢 Baja | **Dashboard** | Conecta `pronostico_7dias.csv` a Power BI para visualización diaria |